## Observations

- The dataset is composed by an id, a target and a large number of locus columns
- The locus columns are named as `chr{chromosome}_{locus}` where chromosome is the chromosome number and locus is the position in the chromosome
- All numeric columns are integers and its values are in the range of int8
- The raw dataset has 11.2 MB while the int8 version has 3.6 MB

## Decisions
- Convert all numeric columns to int8
- Drop id column

In [ ]:
import pandas as pd

from covid.constants import LONG_COVID_RAW_DATA_PATH

df = pd.read_csv(LONG_COVID_RAW_DATA_PATH)
df.shape

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
[col for col in df.columns if not str(col).startswith("chr")]

In [ ]:
df.select_dtypes(include="str").columns.to_list()

In [ ]:
# Number of locus in each chromosome
chr_counts = (
    df.columns
    .to_series()
    .str.extract(r"^(chr[^_]+)", expand=False)
    .value_counts()
    .sort_index()
)
chr_counts

In [ ]:
def can_convert_to_int(series: pd.Series) -> bool:
    numeric = pd.to_numeric(series, errors="coerce")

    has_non_numeric = (numeric.isna() & series.notna()).any()
    if has_non_numeric:
        return False

    all_values_are_int = numeric.dropna().mod(1).eq(0).all()
    return all_values_are_int


non_integer_columns = [
    column
    for column in df.columns
    if not can_convert_to_int(df[column])
]

non_integer_columns

In [ ]:
df.select_dtypes(include="number").astype("int8", errors="ignore").info()